# Member 5 - Website research: concentration and recurring activity
This notebook supplies Member 5's two additional research questions and visualizations. The Dash website uses the **same calculations and figure functions**, so the notebook and dashboard cannot drift into different definitions.

**Input:** the authoritative integrated ZIP-year dataset from Member 3, 2022-2025. No source CSVs are overwritten. Default measures are application filings and recorded serious crashes. These are counts, not completed housing or personal-risk measures.

**Sources:** [H5 DOB NOW](https://data.cityofnewyork.us/resource/w9ak-ipjd), [T4 SIRTA](https://data.cityofnewyork.us/resource/xeqp-qz8h). See `docs/data_contract.md` for preparation and coverage.

## 1. Load and validate the shared data
Resolve the repository from either the root or notebooks directory. Read ZIPs as text and validate keys, count measures and source flags before analysis.

In [1]:
import sys
from pathlib import Path
from IPython.display import display, Markdown
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "app" / "data.py").is_file():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from app.data import load_dataset, build_report, Selection, BENCHMARK_NOTE, dataset_fingerprint
from app.charts import concentration_figure, recurrence_figure

data = load_dataset()
report = build_report(data, Selection())
print(f"Snapshot {dataset_fingerprint()}: {len(data):,} ZIP-years, {data['zip_code'].nunique()} ZIPs")
display(data.head())
display(report.coverage)

Snapshot ead0593db0f1: 881 ZIP-years, 222 ZIPs


,zip_code,year,borough,zip_latitude,zip_longitude,has_housing_data,has_crash_data,filing_count,new_construction_filings,permit_issued_count,...,night_crash_share,night_crash_count,crashes_with_known_time,is_outlier_housing,is_outlier_transport,crashes_per_100_filings,filing_change_yoy,crash_change_yoy,crashes_next_year,severe_injuries_per_1000_units
0,10001,2022,MANHATTAN,40.750314,-73.995511,True,True,2659.0,9.0,1260.0,...,0.176471,3,17,True,False,0.64,NaN,NaN,26.0,0.173
1,10001,2023,MANHATTAN,40.750314,-73.995511,True,True,2567.0,18.0,1124.0,...,0.307692,8,26,True,False,1.01,-92.0,9.0,21.0,0.336
2,10001,2024,MANHATTAN,40.750314,-73.995511,True,True,2650.0,8.0,1201.0,...,0.333333,7,21,True,False,0.79,83.0,-5.0,18.0,0.564
3,10001,2025,MANHATTAN,40.750314,-73.995511,True,True,2532.0,33.0,833.0,...,0.222222,4,18,True,False,0.71,-118.0,-3.0,NaN,0.398
4,10002,2022,MANHATTAN,40.716824,-73.987730,True,True,1334.0,35.0,598.0,...,0.652174,15,23,False,False,1.72,NaN,NaN,31.0,0.289


{'rows': 881,
 'unknown_housing_rows': 4,
 'structural_crash_zeros': 179,
 'approximate_zip_filings': 173,
 'missing_map_zips': ['10044'],
 'known_housing_rows': 877}

## 2. Benchmark definition and missingness
Calculate annual citywide quartiles **before** filtering boroughs, years or ZIPs. Use only housing-covered ZIP-years for the paired-domain comparisons; preserve other rows in dashboard totals and downloads.

The high group is positive and at or above the 75th percentile. Ties are retained, so it may not contain exactly 25% of ZIPs. Zero values are not labelled high even if the percentile threshold is zero. Quartiles are descriptive groups, not treatment assignments or danger ratings.

When combining years, a ZIP can contribute once per year: the denominator is **ZIP-years**, not distinct residents or unique ZIPs. Structural zero crashes mean no retained SIRTA record, not proof of no real-world crashes. Unknown housing is never zero.

In [2]:
print(BENCHMARK_NOTE)
display(report.concentration)

Groups use each year's NYC-wide, housing-covered ZIPs, before your filters. High means a positive value at or above the 75th percentile; ties stay together, so the high group is not necessarily exactly 25% of ZIPs.


,zip_years,transport_events,event_share,record_share
housing_quartile,,,,
Q1 / lowest,220,233,0.022134,0.250855
Q2,218,1861,0.176784,0.248575
Q3,219,4148,0.394034,0.249715
Q4 / highest,220,4285,0.407049,0.250855


## 3. RQ1 - What share of recorded serious crashes falls in the highest-filing activity group?
The grouped bar chart compares each housing group's share of retained serious crashes against its share of selected, housing-covered ZIP-years. The event denominator excludes unknown-housing ZIP-years so every included event can be assigned a housing group.

This is a concentration comparison, not a population-adjusted risk rate. It does not show that filings caused crashes.

In [3]:
figure_rq1 = concentration_figure(report)
figure_rq1.show(renderer="plotly_mimetype")
top = report.concentration.loc["Q4 / highest"]
display(Markdown(
    f"**Finding:** the highest-filing group represents **{top['record_share']:.1%}** of housing-covered ZIP-years "
    f"and **{top['event_share']:.1%}** of their recorded serious crashes. "
    "This measures concentration across places/years, not a construction effect or individual risk."
))

**Finding:** the highest-filing group represents **25.1%** of housing-covered ZIP-years and **40.7%** of their recorded serious crashes. This measures concentration across places/years, not a construction effect or individual risk.

## 4. RQ2 - Which ZIPs repeatedly meet both high-filing and high-crash thresholds?
For each year, separately compare filings and serious crashes to their annual citywide 75th-percentile thresholds. A cell is high only when **both** positive measures meet their thresholds.

The heatmap shows up to 20 ZIPs ordered by high-activity years, then total filings and ZIP. Pale cells are not jointly high; blank cells are unknown or absent. A recurring ZIP meets both conditions in at least two selected years. Changing the selected years changes the recurrence window, not the underlying annual benchmarks.

In [4]:
figure_rq2 = recurrence_figure(report)
figure_rq2.show(renderer="plotly_mimetype")
recurring = report.recurrence[report.recurrence["joint_years"] >= 2]
display(Markdown(
    f"**Finding:** **{len(recurring)} ZIPs** meet both high-activity thresholds in at least two years. "
    "The table gives the actual counts and observed years. Repeated co-occurrence is a screening signal, not causal evidence."
))
display(report.recurrence.head(20))

**Finding:** **32 ZIPs** meet both high-activity thresholds in at least two years. The table gives the actual counts and observed years. Repeated co-occurrence is a screening signal, not causal evidence.

,zip_code,borough,observed_years,joint_years,housing_total
16,10019,MANHATTAN,4,4,10627.0
33,10036,MANHATTAN,4,4,8708.0
122,11101,QUEENS,4,4,7845.0
138,11211,BROOKLYN,4,4,6856.0
147,11220,BROOKLYN,4,4,6322.0
134,11207,BROOKLYN,4,4,5713.0
133,11206,BROOKLYN,4,4,5506.0
1,10002,MANHATTAN,4,4,5190.0
24,10027,MANHATTAN,4,4,4760.0
192,11385,QUEENS,4,4,4401.0


## 5. Verify the research output
Reconcile denominators, retain missingness, and ensure high-activity years never exceed the number of observed years.

In [5]:
known = data[data["has_housing_data"]]
assert report.concentration["zip_years"].sum() == len(known)
assert report.concentration["transport_events"].sum() == known["crash_count"].sum()
assert np.isclose(report.concentration["event_share"].sum(), 1)
assert report.recurrence["joint_years"].le(report.recurrence["observed_years"]).all()
assert report.rows.loc[~report.rows["has_housing_data"], "filing_count"].isna().all()
print("Member 5 research checks passed.")

Member 5 research checks passed.


## 6. Website handoff, limitations and contribution
Run the app from the repository root with `python -m app.app`. It provides multiple dropdowns, a strict documented search mode, Generate Report, reset, five interactive chart types, story text, coverage notices and CSV download of the last applied selection.

The research is limited by incomplete housing coverage, approximate ZIP assignment, serious-crash-only coverage, window-defined housing reports, repeated ZIP-years, spatial dependence and missing population/traffic exposure. The map uses source-mean points, not postal boundaries; coordinate-less ZIPs remain in other charts. Correlation is not causation.

For NYC, use the results to prioritize further street-level investigation alongside traffic volume, population, actual work dates and road-design evidence. Do not rank personal danger or blame development from these counts.

Member 5 implementation was AI-assisted at the user's request; see `docs/prompts.md`. Render configuration is prepared, but public deployment and a second-device check require the user's hosting authorization and remain pending. The deployment guide is in the root README and Member 5 handoff.